# Sen1Floods11 — Full MLOps Pipeline (Colab)

Runs the entire reproducible pipeline end-to-end:

1. Clone repo + install pinned deps
2. Authenticate ClearML + Google Cloud
3. Download Sen1Floods11 hand-labeled split
4. Train SegFormer (or download pretrained checkpoint to skip training)
5. Calibrate the empirical ambiguity band → Figure A
6. Run cascade benchmark → Figure B + system comparison table
7. Inspect ClearML Task IDs (anchor the report numbers)

**Runtime requirement:** GPU runtime (Runtime → Change runtime type → T4 or L4).
Full training takes ~45 min on L4. If you want to skip training, run the
`download-model` cell instead — calibration + benchmark are <5 min on CPU.

Every cell maps to a single `make` target so the report can cite the exact
command used.


## 1. Clone repo & install pinned dependencies

In [ ]:
REPO_URL  = 'https://github.com/<your-org>/<your-repo>.git'   # ← edit me
REPO_PATH = '/content/group_project'

import os, subprocess, sys
if not os.path.exists(REPO_PATH):
    subprocess.run(['git', 'clone', REPO_URL, REPO_PATH], check=True)
%cd $REPO_PATH
!git log -1 --oneline

In [ ]:
# Pinned versions (mlops/requirements.txt) — see REPRODUCING.md.
!pip install -q -r mlops/requirements.txt
import torch, transformers, clearml
print(f'torch={torch.__version__}, transformers={transformers.__version__}, '
      f'clearml={clearml.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Authenticate ClearML

Get your access keys from **app.clear.ml → Settings → Workspace → Create new credentials**.
Paste them into the cell below (they go to env vars, never to disk).

In [ ]:
import os
from getpass import getpass

os.environ['CLEARML_API_HOST']   = 'https://api.clear.ml'
os.environ['CLEARML_WEB_HOST']   = 'https://app.clear.ml'
os.environ['CLEARML_FILES_HOST'] = 'https://files.clear.ml'
os.environ['CLEARML_API_ACCESS_KEY'] = getpass('ClearML access key: ')
os.environ['CLEARML_API_SECRET_KEY'] = getpass('ClearML secret key: ')

# Sanity check — should print your workspace info.
from clearml.backend_api import Session
s = Session()
print('ClearML connected. User:', s.get_decoded_token().get('identity', {}).get('user'))

## 3. Authenticate Google Cloud (for Sen1Floods11 dataset access)

In [ ]:
from google.colab import auth
auth.authenticate_user()
print('GCS authenticated ✓')

## 4. Download Sen1Floods11 hand-labeled split (~2 GB, 3–5 min)

In [ ]:
!make download-data

import os
for d in ['data/sen1floods11/S1', 'data/sen1floods11/Labels', 'data/sen1floods11/splits']:
    print(f'  {d}: {len(os.listdir(d))} files')

## 5a. (Option A) Train SegFormer from scratch

Reproduces the report's 0.668 test IoU number. ClearML logs everything
automatically — metrics, plots, hyperparameters, git state, console output.
Takes ~45 min on L4, ~90 min on T4.

**Skip this cell and use 5b instead if you don't want to wait.**

In [ ]:
!make train-segformer \
    S1_DIR=data/sen1floods11/S1 \
    LABEL_DIR=data/sen1floods11/Labels \
    SPLITS_DIR=data/sen1floods11/splits

## 5b. (Option B) Skip training — pull the pretrained checkpoint

Use this if you only want to validate the cascade benchmark and don't
need a fresh ClearML training task. Pulls the published checkpoint from
Hugging Face.

In [ ]:
!make download-model

## 6. Calibrate the empirical ambiguity band → Figure A

Walks the test set, bins every pixel by `|VV − (−13.45)|`, derives the
distance-to-threshold range where classical accuracy stays below 0.85.
That range is the empirical ambiguity band the cascade uses to decide
which pixels need the deep model.

In [ ]:
!make calibrate \
    S1_DIR=data/sen1floods11/S1 \
    LABEL_DIR=data/sen1floods11/Labels \
    SPLITS_DIR=data/sen1floods11/splits

In [ ]:
# Show Figure A inline.
from IPython.display import Image, display
display(Image('mlops/figures/ambiguity_calibration.png'))

import json
cal = json.load(open('mlops/calibration.json'))
print(f"Empirical ambiguity band: ±{cal['band_edge_db']:.2f} dB")
print(f"  → cascade routes pixels with VV ∈ "
      f"[{cal['db_threshold'] - cal['band_edge_db']:.2f}, "
      f"{cal['db_threshold'] + cal['band_edge_db']:.2f}] dB to deep model")

## 7. Run the cascade benchmark → Figure B + system comparison table

Compares 3 strategies (classical-only, deep-only, cascade) on test + Bolivia
splits, sweeping ambiguity-band widths and Otsu disagreement on/off.
Outputs the system comparison table that goes straight into the report.

In [ ]:
!make benchmark \
    S1_DIR=data/sen1floods11/S1 \
    LABEL_DIR=data/sen1floods11/Labels \
    SPLITS_DIR=data/sen1floods11/splits \
    SEGFORMER_CKPT=checkpoints/segformer_flood_best.pt

In [ ]:
# Show Figure B inline.
from IPython.display import Image, Markdown, display
display(Image('mlops/figures/figure_b_tradeoff.png'))

# Print the auto-generated system comparison table.
with open('mlops/results/system_comparison.md') as f:
    display(Markdown(f.read()))

In [ ]:
# Headline rows from the raw CSV.
import pandas as pd
df = pd.read_csv('mlops/results/benchmark.csv')
print('Test set:')
print(df[df['split'] == 'test'][['strategy', 'IoU', 'F1', 'frac_deep', 'wall_seconds']]
      .to_string(index=False))
print('\nBolivia held-out:')
print(df[df['split'] == 'bolivia'][['strategy', 'IoU', 'F1', 'frac_deep', 'wall_seconds']]
      .to_string(index=False))

## 8. Capture ClearML Task IDs for the report

These IDs anchor every published number to a specific reproducible run.
Paste them into `REPRODUCING.md` before submission.

In [ ]:
from clearml import Task

def latest(project: str):
    tasks = Task.get_tasks(project_name=project)
    if not tasks:
        return None
    return sorted(tasks, key=lambda t: t.data.created)[-1]

for project in [
    'Sen1Floods11/Training',
    'Sen1Floods11/Calibration',
    'Sen1Floods11/Benchmark',
]:
    t = latest(project)
    if t is None:
        print(f'{project:<32}: no tasks yet')
        continue
    print(f'{project:<32}: {t.id}   ({t.name})')
    print(f'  → {t.get_output_log_web_page()}')

## 9. (Optional) Run the cascaded inference pipeline on a single scene

Demonstrates end-to-end: tile → classical fast-pass → deep refinement on
uncertain regions → stitch. Lineage from training Task → inference output
is fully captured in the ClearML UI.

In [ ]:
# Plug in the IDs printed by the cell above.
MODEL_TASK_ID       = ''   # ← from Sen1Floods11/Training
CALIBRATION_TASK_ID = ''   # ← from Sen1Floods11/Calibration

import os
os.environ['MODEL_TASK_ID']       = MODEL_TASK_ID
os.environ['CALIBRATION_TASK_ID'] = CALIBRATION_TASK_ID

if MODEL_TASK_ID and CALIBRATION_TASK_ID:
    !make pipeline
else:
    print('Set MODEL_TASK_ID and CALIBRATION_TASK_ID above first.')

## Done

Generated assets (also stored as ClearML artifacts):

- `mlops/figures/ambiguity_calibration.png` — **Figure A**
- `mlops/figures/figure_b_tradeoff.png`     — **Figure B**
- `mlops/results/system_comparison.md`      — **headline table for the report**
- `mlops/results/benchmark.csv`             — raw per-strategy numbers

All ClearML Tasks are visible at https://app.clear.ml under the
`Sen1Floods11/...` projects.